# PuB$_4$ EFG point charge calculation

In [110]:
from pymatgen.core import Structure
import numpy as np
import pyEFGPointCharge
%matplotlib notebook
from isotopeDict import isotope_data_dict


## Load cif, prepare calc

In [111]:
# load a cif file and create a pymatgen.core.structure.Structure object with the info from said file.
#cif_file_name = 'TmVO4_I41amdS_Milligan_1952_Crystalstructureheavy_ICSD_15602.cif'
# https://doi.org/10.1021/j150493a028

#cif_file_name = 'TmVO4_I41amdZ_Chakoumakos_1994_CrystalStructureRefinements_ICSD_78081.cif'
#https://doi.org/10.1006/jssc.1994.1091

cif_file_name = 'PuB4_Sam.cif'
#cif from Sam Cary

#cif_file_name = 'TmVO4_I41aZ_Bandiello_2020_PhaseBehaviorTmVO4_ICSD_17192.cif'
#https://doi.org/10.1021/acs.inorgchem.0c00147

PuB4_structure = Structure.from_file(cif_file_name)
print(PuB4_structure)


Full Formula (Pu4 B16)
Reduced Formula: PuB4
abc   :   7.084700   7.084700   3.990000
angles:  90.000000  90.000000  90.000000
Sites (20)
  #  SP          a        b      c
---  ----  -------  -------  -----
  0  Pu    0.31335  0.18665  0
  1  Pu    0.81335  0.31335  0
  2  Pu    0.68665  0.81335  0
  3  Pu    0.18665  0.68665  0
  4  B     0.035    0.177    0.5
  5  B     0.823    0.035    0.5
  6  B     0.965    0.823    0.5
  7  B     0.177    0.965    0.5
  8  B     0.535    0.323    0.5
  9  B     0.677    0.535    0.5
 10  B     0.465    0.677    0.5
 11  B     0.323    0.465    0.5
 12  B     0.09     0.41     0.5
 13  B     0.59     0.09     0.5
 14  B     0.91     0.59     0.5
 15  B     0.41     0.91     0.5
 16  B     0        0        0.199
 17  B     0.5      0.5      0.801
 18  B     0        0        0.801
 19  B     0.5      0.5      0.199


## Calculation

In [112]:
sphere_radius = 20


### Pu

In [114]:
# instantiate the classes and define the relevant attributes therein.

# input_parameters is a class within the imported pyEFGPointCharge module
input_pars_11B1 = calc_EFG_point_charge.input_parameters()

##########################
# general input parameters
##########################
# path to cif file
input_pars_11B1.cif_file_name = cif_file_name

# create a supercell if one is required to define the magnetic structure
# the supercell_dimensions parameter should be a list of form [Na, Nb, Nc],
# where Na, Nb, and Nc are integers that represent the number of unit cells
# in the crystallographic a, b, and c directions, respectivly, contained in
# the supercell. For example, [1, 1, 2] will create a supercell that is 
# 1 x 1 x 2 unit cells in size. Note: creating a supercell modifies the
# lattice as well, therefore one must specify the magnetic_site_indices,
# probe indices, etc. in the supercell. see the following for deteails
# dipole_hyperfine_lattice_sum/ipynb_examples/LiCuVO4/
# LiCuVO4_dipole_hyperfine_calc_2021-08-30.ipynb
input_pars_11B1.supercell_dimensions = [1, 1, 1]

# choose the radius around which we will include spins in angstroms with 
# respect to the probe position (float)
input_pars_11B1.sphere_radius = sphere_radius

# define the probe site number (nuclear spin site) (int)
input_pars_11B1.probe_site_index = 17

# for performing large calculations, we want to attempt to keep enough 
# memory. To do this we'll trash some arrays that we don't need
# running on 32bit python would cause memory allocation errors, so to 
# make the program more portable, we can allow for that memory to be 
# freed by the garbage collector by setting (no longer needed arrays 
# to None)
input_pars_11B1.delete_plotting_arrays = False

# probe nucleus identifying dictionary key (from isotope_data_dict) (string)
# e.g. '11B'
input_pars_11B1.probe_nucleus = "11B"

# or if you want to set a custom nuclear quadrupole moment, do that here and
# the default for the probe nucleus will be overridden
input_pars_11B1.manual_nuclear_quad_moment = None

# define the charge sites by index in the cif file (list of ints)
# Note: that site indices in pymatgen start at zero.
# Further Note: if creating a supercell, one must include all magetic 
# site indices in the supercell. therefore the user must first know the 
# site indices.
input_pars_11B1.charge_site_indices = [i for i in range(20)]

# charges should be set to be a numpy array of shape = (len(charge_site_indices),), 
# the order of the charges should be the same as the charge_site_indices. units 
# are in multiples of the elementary charge e.
# Example:
# define antiferromagnetic order on the  magnetic sites with indices 0 and 1,
# so above charge_site_indices = [0, 1], and there is a charge of -2 on site 0
# and a +1 charge on site 1, then charges = np.array([-2.0, 1.0])
input_pars_11B1.first_cell_charges = [4.,  4.,  4.,  4.,
                                     -1., -1., -1., -1.,
                                     -1., -1., -1., -1.,
                                     -1., -1., -1., -1.,
                                     -1., -1., -1., -1.]

# Sternheimer factor (need to implement this, but for now just leave the variable as a placeholder)
input_pars_11B1.gamma_sternheimer = 0

results_11B1 = calc_EFG_point_charge.results()
# a class for returning results of the calculation
# all results are in cartesian coordinates!!!
#class results:
#    # the structure object created from the cif file
#    structure = None
#    # a matrix representing the the lattice unit cell as row vectors in 
#    # cartesian coordinates (angstroms)
#    lattice = None
#    # Charge positions (angstroms)
#    charge_positions = None
#    # charges, in units of the elementary charge e
#    charges = None
#    # raw EFG tensor, ie not diagonalized (V m^-2)
#    V_raw = None
#    # raw eigenvalues of the EFG tensor, ordered such that 
#    # |V_xx| <= |V_yy| <= |V_zz|; where x, y, and z no longer correspond to the 
#    # cartesian axes, but indicate the directions of the principal axes of the
#    # EFG tensor (V m^-2)
#    V_aa = None
#    # eigenvectors of the EFG tensor  (principal axes directions)
#    # (normalized cartesian coordinates, a matrix of column vectors, 
#    # order is the same as the V_aa)
#    principal_axes = None
#    # Quadrupole coupling constants (eigenvalues V_raw_evals with the appropriate
#    # multiplicative factors, including the sternheimer antishielding factor if
#    # provided)
#    QCC_MHz = None

###############################################################################

# run the calc
pyEFGPointCharge.calc_EFG_point_charge(input_pars_11B1, results_11B1)

# if we kept the vectors for plotting, then say how many spins we included in the calculation
if not input_pars_11B1.delete_plotting_arrays:
    N_charges = results_11B1.charge_positions.shape[0]
    print('calculated the EFG tensor via lattice sum for',
          '{:e}'.format(N_charges),
          'charges')
    
# print the results (all in Cartesian coordinates)

print('probe position =')
print(results_11B1.probe_position)
print('')
# spin-lattice magnetization
print('M (mu_B/atom) =')
print(results_Pu.M)
print('')
# dipole hyperfine coupling tensor
print('A (T/mu_B) =')
print(results_Pu.A)
print('')
# internal hyperfine field at the probe site
print('B (T) =')
print(results_Pu.B)


#    structure = Non
#    lattice = None
#    charge_positions = None
#    charges = None
#    V_raw = None
#    V_aa = None
#    principal_axes = None
#    QCC_MHz = None



# display a visualization (note, should only do this for less than approx
# 50 atoms or it is not helpful

#%matplotlib notebook

# define the point-inclusion limits as a tuple
minmax=17
x_min_rel = -minmax
x_max_rel = minmax
y_min_rel = -minmax
y_max_rel = minmax
z_min_rel = -minmax
z_max_rel = minmax
pos_limits = (x_min_rel, x_max_rel, y_min_rel, y_max_rel, z_min_rel, z_max_rel)

# set the axes limits as a tuple
axes_min = -minmax
axes_max = minmax
ax_range = (axes_min, axes_max, axes_min, axes_max, axes_min, axes_max)

# uncommment to execute the visualization, also make sure delete_plotting_arrays = False
pyDipoleHF.visualization(input_parameters=input_pars_Pu,
                         results=results_Pu,
                         relative_position_limits=pos_limits,
                         axes_range=None,
                         elev=0.,
                         azim=0.,
                         plot_sphere=True,
                         plot_lattice_vectors=True)


NameError: name 'pyDipoleHF' is not defined

## summary

check these against the literature

In [6]:
A1_small_ball = results_V1.A
A1_small_ball


array([[-3.86553435e-02,  9.49171218e-18, -3.98391422e-17],
       [ 9.49171218e-18, -3.86553435e-02,  3.46944695e-17],
       [-3.98391422e-17,  3.46944695e-17,  7.73106870e-02]])

In [7]:
A1_Wang


array([[-0.0336,  0.    ,  0.    ],
       [ 0.    , -0.0336,  0.    ],
       [ 0.    ,  0.    ,  0.0671]])

In [8]:
A1_Wang/A1_small_ball


array([[ 0.86922006,  0.        , -0.        ],
       [ 0.        ,  0.86922006,  0.        ],
       [-0.        ,  0.        ,  0.86792658]])

In [9]:
# 150 AA radius ball results
#A_Milligan = np.array(
#[[-3.77360137e-02, 0, 0],
# [0, -3.77360137e-02,  0],
# [0,  0,  7.54720275e-02]])
#
#A_Chakoumakos = np.array(
#[[-3.66980236e-02, 0, 0],
# [0, -3.66980236e-02, 0],
# [0, 0,  7.33960473e-02]])
#
#A_Bandiello_I41amdZ = np.array(
#[[-3.64857613e-02, 0, 0],
# [0, -3.64857613e-02, 0],
# [0, 0,  7.29715225e-02]])
#
#A_Bandiello_I41aZ = np.array(
#[[1.37702560e-02,  0, 0],
# [0,  1.37702560e-02, 0],
# [0, 0, -2.75405119e-02]])


In [10]:
results_V1.positions

array([[[-7.10636132e-16],
        [-5.31472500e+00],
        [ 2.35230000e+00]],

       [[-4.28927822e-16],
        [-1.77157500e+00],
        [-2.35230000e+00]],

       [[-3.54315000e+00],
        [-1.77157500e+00],
        [-7.84100000e-01]],

       [[ 3.39268623e-16],
        [-1.77157500e+00],
        [ 1.01933000e+01]],

       [[-3.26537910e-16],
        [-5.31472500e+00],
        [ 8.62510000e+00]],

       [[-4.48295995e-17],
        [-1.77157500e+00],
        [ 3.92050000e+00]],

       [[-3.54315000e+00],
        [-1.77157500e+00],
        [ 5.48870000e+00]],

       [[ 1.56849177e-15],
        [ 8.85787500e+00],
        [ 2.35230000e+00]],

       [[ 4.28927822e-16],
        [ 1.77157500e+00],
        [ 2.35230000e+00]],

       [[ 7.10636132e-16],
        [ 5.31472500e+00],
        [-2.35230000e+00]],

       [[ 4.48295995e-17],
        [ 1.77157500e+00],
        [-3.92050000e+00]],

       [[-3.54315000e+00],
        [ 1.77157500e+00],
        [ 7.84100000e-01]],

    

## development stuff

### diagonalization


In [96]:
r = np.array([0.1, 0.125, 0.3])
d = np.linalg.norm(r)

# extract the x, y, and z (cartesian coordinates) components of the r_vecs
x = r[0]
y = r[1]
z = r[2]

# calculate the independent hyperfine coupling tensor elements
xx = (3*x**2 - d**2)/d**5
xy = 3*x*y/d**5
xz = 3*x*z/d**5
yy = (3*y**2 - d**2)/d**5
yz = 3*y*z/d**5
zz = (3*z**2 - d**2)/d**5

a = np.array([[xx, xy, xz],
              [xy, xx, yz],
              [xz, yz, zz]])
print(a)


[[-18.83524268   8.24901139  19.79762734]
 [  8.24901139 -18.83524268  24.74703418]
 [ 19.79762734  24.74703418  33.95843023]]


In [97]:
evals, evecs = np.linalg.eig(a)

In [99]:
evals, evecs

(array([ 50.38765148, -25.43445179, -28.66525481]),
 array([[ 2.95914302e-01,  9.48683298e-01,  1.11511102e-01],
        [ 3.52629066e-01,  4.36228898e-16, -9.35763187e-01],
        [ 8.87742906e-01, -3.16227766e-01,  3.34533305e-01]]))

In [102]:
idx = np.argsort(np.abs(evals))

In [103]:
evals[idx]

array([-25.43445179, -28.66525481,  50.38765148])

In [109]:
evecs[:,idx]

array([[ 9.48683298e-01,  1.11511102e-01,  2.95914302e-01],
       [ 4.36228898e-16, -9.35763187e-01,  3.52629066e-01],
       [-3.16227766e-01,  3.34533305e-01,  8.87742906e-01]])

In [104]:
help(np.linalg.eig)

Help on function eig in module numpy.linalg:

eig(a)
    Compute the eigenvalues and right eigenvectors of a square array.
    
    Parameters
    ----------
    a : (..., M, M) array
        Matrices for which the eigenvalues and right eigenvectors will
        be computed
    
    Returns
    -------
    w : (..., M) array
        The eigenvalues, each repeated according to its multiplicity.
        The eigenvalues are not necessarily ordered. The resulting
        array will be of complex type, unless the imaginary part is
        zero in which case it will be cast to a real type. When `a`
        is real the resulting eigenvalues will be real (0 imaginary
        part) or occur in conjugate pairs
    
    v : (..., M, M) array
        The normalized (unit "length") eigenvectors, such that the
        column ``v[:,i]`` is the eigenvector corresponding to the
        eigenvalue ``w[i]``.
    
    Raises
    ------
    LinAlgError
        If the eigenvalue computation does not converg